# Taller 03: Problemas de Satisfacción de Restricciones 

### Grupo: Natalia Carpintero, Paula Núñez e Isabella Arrieta.

**Objetivo:** Diseñe e implemente un algoritmo BT-FC-DO para la solución de Problemas de Satisfacción de Restricciones aplicado a coloreado de grafos. El grafo debe estar definido por una matriz de adyacencias (1: conectado, 0:no conectado) simétrica de tamaño NxN (N < 21). 

In [16]:
import re
import math
import colorsys
import numpy as np
import matplotlib.pyplot as plt

# ---------------------------
# Validacion
# ---------------------------
def validar_tamano(n):
    if not (1 <= n < 21):
        raise ValueError(f"N debe ser menor que 21 (recibido: {n})")
# ---------------------------
# Carga de la matriz
# ---------------------------
def leer_matriz_desde_archivo(ruta):
    with open(ruta, 'r', encoding='utf-8') as archivo:
        lineas = []
        for linea in archivo:
            linea = linea.strip()
            if not linea or linea.startswith('#'):
                continue
            fila = [int(x) for x in re.split(r'[\s,;]+', linea) if x]
            lineas.append(fila)
    if not lineas:
        raise ValueError('El archivo no contiene una matriz de adyacencias válida.')
    matriz = np.array(lineas, dtype=int)
    if matriz.shape[0] != matriz.shape[1]:
        raise ValueError('La matriz debe ser cuadrada.')
    if not validar_tamano(matriz.shape[0]):
        return None
    if not np.all((matriz == 0) | (matriz == 1)):
        raise ValueError('La matriz solo puede contener 0 o 1.')
    if not np.allclose(matriz, matriz.T):
        raise ValueError('La matriz debe ser simétrica.')
    return matriz

# ---------------------------
# Generación aleatoria
# ---------------------------
def generar_grafo_aleatorio(n, probabilidad=0.35, semilla=42):
    if not validar_tamano(n):
        return None
    rng = np.random.default_rng(semilla)
    matriz = np.zeros((n, n), dtype=int)
    for i in range(n):
        for j in range(i + 1, n):
            if rng.random() < probabilidad:
                matriz[i, j] = 1
                matriz[j, i] = 1
    return matriz

# ---------------------------
# Algoritmo BT-FC-DO
# ---------------------------
def bt_fc_do_coloreo(matriz, num_colores=None):
    n = matriz.shape[0]
    if num_colores is None:
        for k in range(1, n + 1):
            solucion = bt_fc_do_coloreo(matriz, k)
            if solucion is not None:
                return solucion, k
        return None, None

    dominios = {v: list(range(num_colores)) for v in range(n)}

    def propagar(asignacion, dominios_actuales, vertice_nuevo, color_nuevo):
        dominios_nuevos = {k: list(v) for k, v in dominios_actuales.items()}
        for u in range(n):
            if u in asignacion or u == vertice_nuevo:
                continue
            if matriz[vertice_nuevo, u] == 1 and color_nuevo in dominios_nuevos[u]:
                dominios_nuevos[u].remove(color_nuevo)
                if not dominios_nuevos[u]:
                    return None
        return dominios_nuevos

    def resolver(asignacion, dominios_actuales):
        if len(asignacion) == n:
            return asignacion.copy()

        no_asignados = [v for v in range(n) if v not in asignacion]
        v = min(no_asignados, key=lambda x: (len(dominios_actuales[x]), -matriz[x].sum()))

        for color in dominios_actuales[v]:
            nueva_asignacion = asignacion.copy()
            nueva_asignacion[v] = color

            dominios_propagados = propagar(asignacion, dominios_actuales, v, color)
            if dominios_propagados is not None:
                dominios_propagados[v] = [color]
                resultado = resolver(nueva_asignacion, dominios_propagados)
                if resultado is not None:
                    return resultado

        return None

    return resolver({}, dominios)

# ---------------------------
# Visualización
# ---------------------------
def generar_paleta(num_colores):
    if num_colores <= 10:
        base = ['#e74c3c', '#2ecc71', '#3498db', '#f1c40f', '#9b59b6',
                '#1abc9c', '#e67e22', '#95a5a6', '#34495e', '#f39c12']
        return base[:num_colores]
    return [
        '#%02x%02x%02x' % tuple(int(c * 255) for c in colorsys.hsv_to_rgb(i / num_colores, 0.65, 0.9))
        for i in range(num_colores)
    ]

def dibujar_grafo(matriz, asignacion):
    n = matriz.shape[0]

    pos = {}
    for i in range(n):
        angulo = 2 * math.pi * i / n
        pos[i] = (math.cos(angulo), math.sin(angulo))

    num_colores = max(asignacion.values()) + 1
    colores = generar_paleta(num_colores)

    fig, ax = plt.subplots(figsize=(8, 8))

    for i in range(n):
        for j in range(i + 1, n):
            if matriz[i, j] == 1:
                x1, y1 = pos[i]
                x2, y2 = pos[j]
                ax.plot([x1, x2], [y1, y2], color='#555555', alpha=0.8, linewidth=1.4, zorder=1)

    for i in range(n):
        x, y = pos[i]
        color = colores[asignacion[i]]
        ax.scatter(x, y, s=1200, color=color, edgecolors='black', linewidths=1.2, zorder=2)
        ax.text(x, y, str(i), ha='center', va='center', fontsize=12, fontweight='bold', zorder=3)

    ax.set_title(f'Coloración de grafo - BT-FC-DO (k={num_colores} colores)')
    ax.set_aspect('equal')
    ax.axis('off')

    plt.savefig('coloreo.png', dpi=150, bbox_inches='tight')
    plt.show()

# ---------------------------
# Ejecución principal
# ---------------------------
if __name__ == "__main__":
    ruta_archivo = 'grafo.txt'
    try:
        matriz = leer_matriz_desde_archivo(ruta_archivo)
        if matriz is not None:
            print('Matriz cargada desde archivo:', ruta_archivo)
    except FileNotFoundError:
        matriz = generar_grafo_aleatorio(10, probabilidad=0.35, semilla=7)
        print('Archivo no encontrado. Se usó un grafo aleatorio.')

    if matriz is not None:
        solucion, num_colores = bt_fc_do_coloreo(matriz)
        if solucion is None:
            print('No existe una coloración válida para este grafo.')
        else:
            print('Número mínimo de colores usados:', num_colores)
            print('Asignación por nodo:', solucion)
            dibujar_grafo(matriz, solucion)